# VDM for generative modeling

The notebook implements the **Variational Diffusion Model (VDM)** — a special case of the SDE notebook with three substitutions:

1. The variational distribution uses a **fixed VP scalar schedule** instead of learned networks: $\alpha(y,t)=\alpha(t)\,y$ and $\beta(y,t)=\beta(t)$, derived from a linear log-SNR schedule with two learnable endpoints.
2. The diffusion coefficient $\sigma(x,t)=\sigma(t)$ is also fixed by the schedule via $\sigma^2(t)=\alpha^2(t)\,\frac{d}{dt}[\beta^2/\alpha^2]$.
3. The drift is set to $f(x,t)=g(x,\widehat{y}(x,t),t)$, where $\widehat{y}(x_t,t)$ is a learned **signal prediction network**.

With these substitutions the ELBO drift-matching term $(f-g)^2/(2\sigma^2)$ reduces exactly to an SNR-weighted prediction loss, giving:
$$
\log p(y)\;\ge\;
\underbrace{\mathbb{E}_{q_1}[\log p(y|x_1)]}_{\text{likelihood}}
-\underbrace{\mathrm{KL}(q_0\|p_0)}_{=\,0}
-\frac{1}{2}\int_0^1 \frac{d\,\mathrm{SNR}(t)}{dt}\,\mathbb{E}_{q_t}\!\left[(y-\widehat{y}(x_t,t))^2\right]dt.
$$

Convention: $t=0$ is the noise prior $p_0=\mathcal{N}(0,1)$, $t=1$ is the data — matching notebooks 01–03.

## Setup

In [20]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.system("wget -q https://raw.githubusercontent.com/olewinther/generative-ode-sde/main/utils.py")
else:
    for path in ['..', '.']:
        if os.path.exists(os.path.join(path, 'utils.py')):
            sys.path.insert(0, os.path.abspath(path))
            break

from utils import *

## GPU

In [21]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU State:', device)

GPU State: cpu


## Network components

Replacing the SDE notebook's learned `AlphaNetwork`, `BetaNetwork`, `DiffusionNetwork`, and `DriftNetwork` with their VDM counterparts:

- **SNRSchedule**: derives $\alpha(t)$, $\beta(t)$, $\sigma(t)$ from $\log\mathrm{SNR}(t)=(1-t)\log\mathrm{SNR}(0)+t\,\log\mathrm{SNR}(1)$ and VP.
- **SignalPredictionNetwork** $\widehat{y}(x_t,t)$: replaces the drift network.
- **VDMDrift** / **VDMBackwardDrift** / **VDMSigma**: wrap the schedule and $\widehat{y}$ into the callables expected by `ForwardPath` / `BackwardPath`.

In [ ]:
import torch.nn as nn


class SNRSchedule(nn.Module):
    """Linear log-SNR schedule with variance preservation."""
    def __init__(self, log_snr_0=-10.0, log_snr_1=10.0, trainable=False):
        super().__init__()
        for name, val in [('log_snr_0', log_snr_0), ('log_snr_1', log_snr_1)]:
            buf = torch.tensor(float(val))
            if trainable:
                setattr(self, name, nn.Parameter(buf))
            else:
                self.register_buffer(name, buf)

    def log_snr(self, t):  return (1 - t) * self.log_snr_0 + t * self.log_snr_1
    def snr(self, t):      return torch.exp(self.log_snr(t))
    def alpha(self, t):    return torch.sqrt(self.snr(t) / (1 + self.snr(t)))
    def beta(self, t):     return torch.sqrt(1 / (1 + self.snr(t)))

    def sigma_sq(self, t):
        # α²·d/dt[β²/α²] = (log_snr_0 - log_snr_1)·β²  [< 0 since log_snr_1 > log_snr_0]
        return (self.log_snr_0 - self.log_snr_1) * self.beta(t) ** 2

    def d_log_alpha_dt(self, t):  return 0.5 * (self.log_snr_1 - self.log_snr_0) * self.beta(t) ** 2
    def d_alpha_dt(self, t):      return self.alpha(t) * self.d_log_alpha_dt(t)
    def d_beta_dt(self, t):       return -self.alpha(t) / self.beta(t) * self.d_alpha_dt(t)
    def d_snr_dt(self, t):        return (self.log_snr_1 - self.log_snr_0) * self.snr(t)


class SignalPredictionNetwork(nn.Module):
    """Predicts y from noisy x_t and time t."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=-1))


def g_vdm(x, y, schedule, t):
    """Variational drift g(x, y, t) with VDM schedule — same formula as the SDE notebook
    but with analytical d_alpha/dt, d_beta/dt instead of autograd."""
    alpha       = schedule.alpha(t)
    beta        = schedule.beta(t)
    d_alpha     = schedule.d_alpha_dt(t)
    d_beta      = schedule.d_beta_dt(t)
    sigma_sq_eff = -schedule.sigma_sq(t)          # σ̃² = -σ² > 0
    score       = -(x - alpha * y) / beta ** 2
    return d_alpha * y + d_beta * (x - alpha * y) / beta + (sigma_sq_eff / 2) * score


class VDMDrift(nn.Module):
    """f(x,t) = g(x, yhat(x,t), t): same formula as SDE notebook's g, with y replaced by yhat."""
    def __init__(self, yhat_net, schedule):
        super().__init__()
        self.yhat_net = yhat_net
        self.schedule = schedule

    def forward(self, x, t):
        yhat = self.yhat_net(x, t)
        return g_vdm(x, yhat, self.schedule, t)


class VDMBackwardDrift(nn.Module):
    """Backward ODE drift for encoding data to prior (mode='ode': no sigma correction)."""
    def __init__(self, yhat_net, schedule):
        super().__init__()
        self.yhat_net = yhat_net
        self.schedule = schedule

    def forward(self, x, t):
        yhat    = self.yhat_net(x, t)
        alpha   = self.schedule.alpha(t)
        beta    = self.schedule.beta(t)
        d_alpha = self.schedule.d_alpha_dt(t)
        d_beta  = self.schedule.d_beta_dt(t)
        return d_alpha * yhat + d_beta * (x - alpha * yhat) / beta


class VDMSigma(nn.Module):
    """Diffusion coefficient σ̃(x,t) = sqrt(-sigma_sq(t)) > 0."""
    def __init__(self, schedule):
        super().__init__()
        self.schedule = schedule

    def forward(self, x, t):
        return torch.sqrt(-self.schedule.sigma_sq(t)).expand_as(x)


class Beta(nn.Module):
    def __init__(self, initial_beta=1e-1, trainable=True):
        super().__init__()
        log_b = torch.log(torch.tensor(initial_beta))
        if trainable:
            self.log_beta = nn.Parameter(log_b)
        else:
            self.register_buffer("log_beta", log_b)

    def forward(self):
        return torch.exp(self.log_beta)


class Likelihood(nn.Module):
    def __init__(self, beta):
        super().__init__()
        self.beta = beta

    def log_prob(self, y, x):
        var = self.beta() ** 2
        return -0.5 * ((y - x) ** 2 / var + torch.log(2 * torch.pi * var))

    def sample(self, x):
        return x + torch.randn_like(x) * self.beta()

## Visualize schedule and learned signal prediction

In [ ]:
def plot_schedule(schedule):
    t_grid = torch.linspace(0, 1, 200).unsqueeze(1)
    with torch.no_grad():
        alpha_vals = [schedule.alpha(t).item() for t in t_grid]
        beta_vals  = [schedule.beta(t).item()  for t in t_grid]
        snr_vals   = [schedule.snr(t).item()   for t in t_grid]

    fig, axes = plt.subplots(3, 1, figsize=(10, 9))
    axes[0].plot(t_grid.numpy(), alpha_vals)
    axes[0].set_title(r"$\alpha(t)$"); axes[0].set_xlabel("t"); axes[0].set_ylabel(r"$\alpha(t)$")
    axes[1].plot(t_grid.numpy(), beta_vals)
    axes[1].set_title(r"$\beta(t)$"); axes[1].set_xlabel("t"); axes[1].set_ylabel(r"$\beta(t)$")
    axes[2].semilogy(t_grid.numpy(), snr_vals)
    axes[2].set_title("SNR(t)"); axes[2].set_xlabel("t"); axes[2].set_ylabel("SNR(t)")
    plt.tight_layout(); plt.show()


def plot_yhat(yhat_net, schedule, num_samples=3):
    t_grid    = torch.linspace(0, 1, 100).unsqueeze(1)
    x_samples = torch.randn(num_samples, 1)

    fig, ax = plt.subplots(figsize=(10, 3))
    for s in x_samples:
        with torch.no_grad():
            vals = [yhat_net(s, t.expand_as(s)).item() for t in t_grid]
        ax.plot(t_grid.numpy(), vals, label=f"x={s.item():.2f}")
    ax.set_title(r"$\widehat{y}(x,t)$"); ax.set_xlabel("t"); ax.set_ylabel(r"$\widehat{y}$"); ax.legend()
    plt.tight_layout(); plt.show()

## VDM ELBO and training loop

The KL term vanishes exactly since $q_0(x|y)=\mathcal{N}(0,1)=p_0(x)$. The drift matching term $(f-g)^2/(2\sigma^2)$ reduces exactly to $\tfrac{1}{2}\frac{d\,\mathrm{SNR}}{dt}(y-\widehat{y})^2$.

In [ ]:
def compute_elbo_vdm(yhat_net, schedule, prior, likelihood, y, t_sample):
    # 1. Likelihood at t=1: x_1 ~ q_1(x|y) = N(alpha(1)*y, beta(1)^2)
    alpha_1 = schedule.alpha(torch.ones_like(t_sample))
    beta_1  = schedule.beta(torch.ones_like(t_sample))
    x_1 = alpha_1 * y + beta_1 * torch.randn_like(y)
    likelihood_term = likelihood.log_prob(y, x_1)

    # 2. KL(q_0 || p_0): q_0(x|y) = N(alpha(0)*y, beta(0)^2) ≈ N(0,1) = p_0
    alpha_0 = schedule.alpha(torch.zeros_like(t_sample))
    beta_0  = schedule.beta(torch.zeros_like(t_sample))
    x_0 = alpha_0 * y + beta_0 * torch.randn_like(y)
    q0_log_prob = -0.5 * (((x_0 - alpha_0 * y) / beta_0) ** 2 + torch.log(2 * torch.pi * beta_0 ** 2))
    kl_divergence = q0_log_prob - prior.log_prob(x_0)

    # 3. Drift matching: 0.5*(f - g)^2 / sigma^2
    #    f(x_t, t) = g(x_t, yhat, t);  g_true = g(x_t, y, t)  — analytical, no autograd
    alpha_t      = schedule.alpha(t_sample)
    beta_t       = schedule.beta(t_sample)
    sigma_sq_eff = -schedule.sigma_sq(t_sample)   # σ̃² > 0

    x_t   = alpha_t * y + beta_t * torch.randn_like(y)
    yhat  = yhat_net(x_t, t_sample)
    f_xt  = g_vdm(x_t, yhat, schedule, t_sample)
    g_xt  = g_vdm(x_t, y,    schedule, t_sample)
    drift_term = 0.5 * (f_xt - g_xt) ** 2 / sigma_sq_eff

    elbo = likelihood_term - kl_divergence - drift_term
    return elbo.mean(), likelihood_term.mean(), kl_divergence.mean(), drift_term.mean()


def training_loop_vdm(yhat_net, schedule, prior, likelihood, data_loader, validation_set,
                      n_epochs=1000, lr=1e-3):
    from collections import deque
    optimizer = torch.optim.Adam(
        list(yhat_net.parameters()) + list(likelihood.beta.parameters()), lr=lr)
    train_history, val_history = deque(maxlen=5), deque(maxlen=5)
    for epoch in range(n_epochs):
        total_elbo = 0.0
        for y_batch in data_loader:
            optimizer.zero_grad()
            t_sample = torch.rand(y_batch.size())
            elbo, *_ = compute_elbo_vdm(yhat_net, schedule, prior, likelihood, y_batch, t_sample)
            total_elbo += elbo.item()
            (-elbo).backward()
            optimizer.step()
        if epoch % 50 == 0 or epoch == n_epochs - 1:
            t_val = torch.rand(validation_set.size())
            elbo_val, ll, kl, dm = compute_elbo_vdm(
                yhat_net, schedule, prior, likelihood, validation_set, t_val)
            train_cur = total_elbo / len(data_loader)
            val_cur = elbo_val.item()
            train_history.append(train_cur)
            val_history.append(val_cur)
            avg_train = sum(train_history) / len(train_history)
            avg_val   = sum(val_history)   / len(val_history)
            print(f"Epoch {epoch:4d} | train {train_cur:.4f} (avg {avg_train:.4f}) | "
                  f"val {val_cur:.4f} (avg {avg_val:.4f}) | "
                  f"ll={ll:.4f}, kl={kl:.4f}, dm={dm:.4f}")
    return yhat_net, schedule

## Create training and validation data

In [28]:
torch.manual_seed(42)
np.random.seed(42)

n_samples, n_val = 1000, 8000

# Choose distribution: 'gaussian', 'laplace', 'laplace_mixture'
training_set_dist = 'laplace_mixture'

if training_set_dist == 'gaussian':
    params = {'mean': torch.tensor(-1.0), 'std': torch.tensor(2.0)}
elif training_set_dist == 'laplace':
    params = {'loc': torch.tensor(0.0), 'scale': torch.tensor(1.0)}
elif training_set_dist == 'laplace_mixture':
    params = {'k': 5, 'spacing': 4.0, 'scale': torch.tensor(1.0)}

training_set = TrainingSetWithLogLikelihood(training_set_dist, params)
training_data, ell_train = training_set.generate_training_data(n_samples)
validation_data, ell_val = training_set.generate_training_data(n_val)
print(f"{training_set_dist} | true log-likelihood: train={ell_train:.4f}, val={ell_val:.4f}")

laplace_mixture | true log-likelihood: train=-2.9829, val=-2.9862


## Run training

In [ ]:
data_loader = torch.utils.data.DataLoader(training_data, batch_size=125, shuffle=True)

schedule  = SNRSchedule(log_snr_0=-10.0, log_snr_1=10.0, trainable=False)
yhat_net  = SignalPredictionNetwork()
prior     = Prior(gaussian_sample, gaussian_log_pdf)
delta     = Beta(initial_beta=1e-1, trainable=True)
likelihood = Likelihood(beta=delta)

t = torch.linspace(0, 1, steps=100)

f_net     = VDMDrift(yhat_net, schedule)
f_net_bwd = VDMBackwardDrift(yhat_net, schedule)
sigma_net = VDMSigma(schedule)

forward_path  = ForwardPath(mode="sde", f_net=f_net, sigma_net=sigma_net, prior=prior)
backward_path = BackwardPath(mode="ode", f_net=f_net_bwd)

plot_schedule(schedule)
plot_yhat(yhat_net, schedule)
visualize_paths_and_marginals(validation_data, t, backward_path, forward_path)

yhat_net, schedule = training_loop_vdm(
    yhat_net, schedule, prior, likelihood,
    data_loader, validation_data, n_epochs=2000, lr=1e-3,
)

plot_schedule(schedule)
plot_yhat(yhat_net, schedule)
visualize_paths_and_marginals(validation_data, t, backward_path, forward_path)